# 03 - Adaptive retrieval depth: baselines and the proposed methods
| Family | Method | Idea |
|---|---|---|
| fixed | top-k | same k for every query |
| rule | relative score / z-score | keep passages above a threshold |
| prior art | **Adaptive-k** (Taguchi et al., 2025) | cut at the largest gap in the sorted similarity scores + buffer |
| prior art | **CAR-lite** (after Xu et al., 2025) | 2-means on scores, cut at cluster boundary |
| **proposed** | **RCAK** - Risk-Controlled Adaptive-k | gradient-boosted *quantile* regression of log(rank of first gold passage) from score-distribution features, then **split-conformal calibration**. Guarantees P(gold rank <= k_hat) >= 1 - alpha under exchangeability; alpha is an interpretable dial. |
| **proposed** | **CGC** - Confidence-Gated Cascade | classifier predicts P(gold in top-3); serve 3 if confident else 20 |
| bound | Oracle | smallest k containing the gold passage |

Features use **only retrieval-score statistics, dense/BM25 agreement and the question wh-word** - no LLM call, so the controller costs milliseconds.

**Outputs:** `csv/strategy_results.csv`, `csv/feature_matrix_sample.csv`

In [1]:
# ---- Environment setup (works in Colab AND in a local venv / Jupyter) ----
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "sentence-transformers", "python-docx", "psutil"], check=False)
    if not pathlib.Path("adaptive_rag").exists():
        from google.colab import files
        print("Upload rag_adaptive_project.zip (the project archive you downloaded)")
        up = files.upload(); z = [k for k in up if k.endswith(".zip")][0]
        zipfile.ZipFile(z).extractall("."); 
        root = [p for p in pathlib.Path(".").iterdir() if (p / "adaptive_rag").exists()][0]; os.chdir(root)
    try:
        from google.colab import drive; drive.mount("/content/drive")      # results persist on your Drive
        os.environ["RAG_RESULTS_DIR"] = "/content/drive/MyDrive/rag_adaptive_results"
    except Exception as e:
        os.environ["RAG_RESULTS_DIR"] = os.path.abspath("results")
    BACKEND = "st"          # real neural embeddings on Colab (GPU recommended: Runtime > Change runtime type > T4)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    sys.path.insert(0, str(root)); os.chdir(root)
    os.environ.setdefault("RAG_RESULTS_DIR", str(root / "results_notebooks"))
    BACKEND = "lsa"         # offline CPU baseline (TF-IDF+SVD). Set to "st" if sentence-transformers can download models.
sys.path.insert(0, os.getcwd())
CFG = dict(backend=BACKEND, st_model="sentence-transformers/all-MiniLM-L6-v2", n_queries=int(os.environ.get("RAG_NQ", 10000)), seed=0)
print("backend:", BACKEND, "| results ->", os.environ["RAG_RESULTS_DIR"])


backend: lsa | results -> /home/claude/rag_adaptive_project/results_notebooks


In [2]:
import numpy as np, pandas as pd
from adaptive_rag import experiment as E
from adaptive_rag.features import feature_groups
from adaptive_rag.utils import results_dir
B = E.prepare_cached(CFG)
B.feat.assign(split=B.split).head(200).to_csv(results_dir("csv")/"feature_matrix_sample.csv", index=False)
feature_groups()

{'dense_scores': ['d_s1',
  'd_gap12',
  'd_gap15',
  'd_gap110',
  'd_mean10',
  'd_std10',
  'd_ratio10',
  'd_entropy'],
 'bm25_scores': ['b_s1', 'b_gap12', 'b_mean10', 'b_ratio10', 'b_entropy'],
 'agreement': ['overlap5', 'overlap10', 'd1_rank_in_bm25', 'b1_rank_in_dense'],
 'query': ['q_words',
  'qt_who',
  'qt_when',
  'qt_where',
  'qt_what',
  'qt_which',
  'qt_how',
  'qt_why',
  'qt_other']}

In [3]:
df, perq, te = E.evaluate(B, return_per_query=True)
df.to_csv(results_dir("csv")/"strategy_results.csv", index=False)
cols = ["strategy","param","mean_k","hit_rate","answer_in_context","mean_ctx_tokens_est"]
display(df[df.serve_retriever=="hybrid_rrf"][cols].round(3).sort_values(["strategy","mean_k"]).reset_index(drop=True))

,strategy,param,mean_k,hit_rate,answer_in_context,mean_ctx_tokens_est
0,CGC (proposed),tau=0.2,9.152,0.731,0.763,1370.369
1,CGC (proposed),tau=0.35,10.808,0.782,0.808,1625.491
2,CGC (proposed),tau=0.5,11.807,0.808,0.833,1785.893
3,CGC (proposed),tau=0.6,12.420,0.818,0.842,1881.009
4,CGC (proposed),tau=0.7,12.998,0.828,0.851,1970.869
5,CGC (proposed),tau=0.8,13.796,0.837,0.859,2095.903
6,CGC (proposed),tau=0.9,14.839,0.846,0.866,2256.631
7,CGC (proposed),tau=0.95,15.855,0.853,0.872,2412.362
8,RCAK (proposed),alpha=0.6,7.482,0.637,0.682,1129.827
9,RCAK (proposed),alpha=0.5,9.588,0.704,0.743,1449.202


### Matched-budget view (headline table)
For each adaptive method we find the fixed top-k with the **same mean number of passages** (linear interpolation along the fixed-k curve) and report the difference.

In [4]:
iso = E.isobudget(B, "hybrid_rrf", n_boot=500); iso.to_csv(results_dir("csv")/"isobudget_hybrid.csv", index=False)
best = iso[iso.metric=="answer_in_context"].sort_values("diff", ascending=False).head(12)
display(best[["strategy","param","mean_k","adaptive_value","fixed_value_at_same_mean_k","diff","ci95_lo","ci95_hi"]].round(3))

,strategy,param,mean_k,adaptive_value,fixed_value_at_same_mean_k,diff,ci95_lo,ci95_hi
58,RCAK (proposed),alpha=0.07,21.103,0.930,0.882,0.048,0.036,0.061
74,CGC (proposed),tau=0.7,12.998,0.851,0.804,0.047,0.034,0.058
76,CGC (proposed),tau=0.8,13.796,0.859,0.814,0.045,0.034,0.056
60,RCAK (proposed),alpha=0.05,22.446,0.936,0.891,0.044,0.034,0.057
72,CGC (proposed),tau=0.6,12.420,0.842,0.798,0.044,0.030,0.056
52,RCAK (proposed),alpha=0.2,19.126,0.909,0.865,0.044,0.028,0.057
54,RCAK (proposed),alpha=0.15,19.250,0.910,0.866,0.044,0.029,0.057
56,RCAK (proposed),alpha=0.1,19.487,0.911,0.868,0.043,0.027,0.056
78,CGC (proposed),tau=0.9,14.839,0.866,0.824,0.042,0.031,0.052
70,CGC (proposed),tau=0.5,11.807,0.833,0.791,0.042,0.028,0.054
